# ✍️ Feedback Prize: English Language Learning : EDA & MultiOutputRegressor

## 📖 Proje Amacı

Kaggle **Feedback Prize - English Language Learning** yarışmasının temel amacı, ana dili İngilizce olmayan öğrencilerin yazdığı makalelerin dil yeterliliğini otomatik olarak puanlamaktır. Modelimiz, bir metni girdi olarak alıp altı farklı analitik kriter (Cohesion, Syntax, Vocabulary, Phraseology, Grammar, Conventions) üzerinden 1.0 ile 5.0 aralığında tahmin yürüten bir **Çoklu Çıktılı Regresyon (Multi-output Regression)** görevi üstlenmektedir.

## 🛠️ Uygulanan NLP ve Modelleme Aşamaları

Proje kapsamında uygulanan teknik iş akışı, verinin ham metin halinden canlı bir tahminleme aracına dönüşene kadar şu sırayı izlemektedir:

### 1. Metin Ön İşleme (Regex ile Veri Temizleme)

Ham metinlerdeki gürültüyü temizlemek için düzenli ifadeler (`regex`) kullanılmıştır. Bu aşamada paragraflar arasındaki gereksiz boşluklar ve satır sonu karakterleri temizlenerek metin standardize edilmiştir.

### 2. Özellik Çıkarımı (TF-IDF Vektörleştirme)

`TfidfVectorizer` kullanılarak metinsel veriler sayısal vektörlere dönüştürülmüştür. Bu yöntemle, kelimelerin hem makale içindeki sıklığı hem de tüm veri seti üzerindeki ayırt ediciliği hesaplanarak istatistiksel bir ağırlıklandırma yapılmıştır.

### 3. Çoklu Çıktı Mimarisi (MultiOutputRegressor)

Projenin en kritik adımı olan bu aşamada `MultiOutputRegressor` kullanılmıştır. Bu yapı, temel regresyon algoritmasını sarmalayarak altı farklı hedef değişkeni (puan türünü) aynı anda tahmin edebilen tek bir güçlü model mimarisi oluşturur.

### 4. Model Eğitimi (Ridge Regression)

`MultiOutputRegressor` içerisinde ana tahminleyici olarak **Ridge Regression** kullanılmıştır. Ridge algoritması, "L2 regülarizasyonu" uygulayarak modelin eğitim verisine aşırı uyum sağlamasını (overfitting) engeller ve daha kararlı tahminler üretilmesini sağlar.

### 5. Uçtan Uca İş Akışı (Scikit-learn Pipeline)

Vektörleştirme (`tfidf`) ve tahmin (`regressor`) adımları bir `Pipeline` yapısı içinde birleştirilmiştir. Bu sayede veri sızıntısı önlenmiş, kodun okunabilirliği artırılmış ve metinden tahmine giden süreç tek bir hamlede otomatize edilmiştir.

### 6. Performans Değerlendirme (Mean Squared Error)

Modelin başarısını ölçmek için `mean_squared_error` metriği kullanılmıştır. Tahmin edilen puanlar ile gerçek puanlar arasındaki farklar hesaplanarak modelin hata payı ve isabet oranı belirlenmiştir.


**Veri Sözlüğü (Puanlama Kriterleri)**

| Sütun Adı | Veri Tipi | Açıklama |
| --- | --- | --- |
| **id** | Metin | Her makale için benzersiz tanımlayıcı. |
| **full_text** | Metin | Analiz edilen makalenin tam içeriği. |
| **cohesion** | Sayısal | **Hedef;** Metnin bağlamsal bütünlüğü ve akıcılığı. |
| **syntax** | Sayısal | **Hedef;** Cümle yapılarının doğruluğu. |
| **vocabulary** | Sayısal | **Hedef;** Kelime dağarcığının zenginliği. |
| **phraseology** | Sayısal | **Hedef;** Deyim ve kalıpların doğal kullanımı. |
| **grammar** | Sayısal | **Hedef;** Dilbilgisi kurallarına uygunluk. |
| **conventions** | Sayısal | **Hedef;** Yazım ve noktalama kurallarına uyum. |

In [1]:
# Veri manipülasyonu ve analizi için kullanılır; tabloları (DataFrame) okumayı ve düzenlemeyi sağlar.
import pandas as pd

# Bilimsel hesaplamalar ve çok boyutlu diziler (matrix) üzerinde hızlı matematiksel işlemler yapmak içindir.
import numpy as np

# Verilerin temel 2 boyutlu grafiklerini (çizgi, sütun, dağılım vb.) oluşturmak için kullanılan temel kütüphanedir.
import matplotlib.pyplot as plt

# Matplotlib tabanlı, daha gelişmiş, estetik ve istatistiksel veri görselleştirmeleri yapmak için kullanılır.
import seaborn as sns

In [2]:
#veriyi oku
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

# EDA

In [3]:
#ilk 5 veriyi göster
train.head()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5
2,00299B378633,"Dear, Principal\n\nIf u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5
3,003885A45F42,The best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0
4,0049B1DF5CCC,Small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5


In [4]:
#ilk 5 veriyi göster
test.head()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."


In [5]:
#son 5 veriyi göster
train.tail()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
3906,FFD29828A873,I believe using cellphones in class for educat...,2.5,3.0,3.0,3.5,2.5,2.5
3907,FFD9A83B0849,"Working alone, students do not have to argue w...",4.0,4.0,4.0,4.0,3.5,3.0
3908,FFDC4011AC9C,"""A problem is a chance for you to do your best...",2.5,3.0,3.0,3.0,3.5,3.0
3909,FFE16D704B16,Many people disagree with Albert Schweitzer's ...,4.0,4.5,4.5,4.0,4.5,4.5
3910,FFED00D6E0BD,Do you think that failure is the main thing fo...,3.5,2.5,3.5,3.0,3.0,3.5


In [6]:
#son 5 veriyi göster
test.tail()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."


In [7]:
#veriler hk bilgi
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3911 entries, 0 to 3910
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   text_id      3911 non-null   object 
 1   full_text    3911 non-null   object 
 2   cohesion     3911 non-null   float64
 3   syntax       3911 non-null   float64
 4   vocabulary   3911 non-null   float64
 5   phraseology  3911 non-null   float64
 6   grammar      3911 non-null   float64
 7   conventions  3911 non-null   float64
dtypes: float64(6), object(2)
memory usage: 244.6+ KB


In [8]:
#0.veriyi gösterir
train['full_text'][0]

"I think that students would benefit from learning at home,because they wont have to change and get up early in the morning to shower and do there hair. taking only classes helps them because at there house they'll be pay more attention. they will be comfortable at home.\n\nThe hardest part of school is getting ready. you wake up go brush your teeth and go to your closet and look at your cloths. after you think you picked a outfit u go look in the mirror and youll either not like it or you look and see a stain. Then you'll have to change. with the online classes you can wear anything and stay home and you wont need to stress about what to wear.\n\nmost students usually take showers before school. they either take it before they sleep or when they wake up. some students do both to smell good. that causes them do miss the bus and effects on there lesson time cause they come late to school. when u have online classes u wont need to miss lessons cause you can get everything set up and go t

In [9]:
import re

def temizle(text):
    # 1. Satır başlarını ve yeni satır karakterlerini boşlukla değiştir
    text = text.replace('\n', ' ').replace('\r', ' ')
    
    # 2. Fazla boşlukları tek boşluğa indir
    text = re.sub(r'\s+', ' ', text)
    
    # 3. Baştaki ve sondaki boşlukları sil
    text = text.strip()
    
    return text

# Uygulama
train['full_text'] = train['full_text'].apply(temizle)

In [10]:
#0.veriyi gösterir
train['full_text'][0]

"I think that students would benefit from learning at home,because they wont have to change and get up early in the morning to shower and do there hair. taking only classes helps them because at there house they'll be pay more attention. they will be comfortable at home. The hardest part of school is getting ready. you wake up go brush your teeth and go to your closet and look at your cloths. after you think you picked a outfit u go look in the mirror and youll either not like it or you look and see a stain. Then you'll have to change. with the online classes you can wear anything and stay home and you wont need to stress about what to wear. most students usually take showers before school. they either take it before they sleep or when they wake up. some students do both to smell good. that causes them do miss the bus and effects on there lesson time cause they come late to school. when u have online classes u wont need to miss lessons cause you can get everything set up and go take a 

In [11]:
# X: Modelin okuyacağı temizlenmiş metinler
x = train['full_text'].apply(temizle) 

# y: Modelin tahmin edeceği 6 farklı puan türü
target_columns = ['cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions']
y = train[target_columns]

In [12]:
# Veri setini, modeli eğitmek için 'eğitim' ve performansını test etmek için 'test' olarak ikiye böler.
from sklearn.model_selection import train_test_split

In [13]:
#  SPLIT (HER ŞEYDEN ÖNCE)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Basit Model Tasarımı
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')), # Metni sayısallaştır
    ('regressor', MultiOutputRegressor(Ridge())) # 6 puanı aynı anda tahmin et
])

# Modeli Eğit
model.fit(x_train, y_train)

,steps,"[('tfidf', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [24]:
#tahmin
tahmin=model.predict(x_test)

In [25]:
# MCRMSE hesaplama (Yarışma Metriği)
def mcrmse(y_true, y_pred):
    rmse_list = np.sqrt(np.mean((y_true - y_pred)**2, axis=0))
    return np.mean(rmse_list)

skor = mcrmse(y_test, tahmin)
print(f"Modelin Ortalama Hata Skoru (MCRMSE): {skor:.4f}")

Modelin Ortalama Hata Skoru (MCRMSE): 0.5618


In [27]:
# Test setindeki veriler için tahmin yapalım
test_preds = model.predict(test['full_text'].apply(temizle))

# Tahminleri bir DataFrame'e dönüştürelim
submission = pd.DataFrame(test_preds, columns=target_columns)

# Yarışma formatı için text_id sütununu ekleyelim
submission.insert(0, 'text_id', test['text_id'])

# Dosyayı kaydedelim
submission.to_csv('submission.csv', index=False)

In [28]:
submission

,text_id,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0000C359D63E,2.772383,2.776561,3.211203,3.057394,2.616791,2.723039
1,000BAD50D026,2.934778,2.807204,2.799301,2.661400,2.798421,3.070569
2,00367BB2546B,3.246366,3.406806,3.322005,3.252313,3.218549,3.234238


In [29]:
import joblib

# Pipeline'ı (Tfidf + Ridge) tek bir dosya olarak kaydediyoruz
joblib.dump(model, 'feedback_prize_model.pkl')


['feedback_prize_model.pkl']

In [30]:
test.head()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."


In [32]:
#0.veriyi gösterir
test['full_text'][1]

"Do you think students would benefit from being able attend classes from home?\n\nYes! its benefit for student who attend classes from home. Because some student want to attend classes from home because they thinks it's very important for them . And they think they can learned fast, and understand than they student who attend classes from school. For example my friend told me that she's attand classes from home it's good for her, because they is some subject she didn't understand when she attend classes from school but when she attend the home classes she good for that subject. she like science she think science its important for her but she didn't understand anything any science but she understand science because she attend classes from home. And some parent doesn't want they are children to attend classes from school, because many christian doesn't want they are children to attend to classes from school because they doesn't believe science and they want they are children to be disres

## 🏁 Sonuç ve Değerlendirme

Bu projede, ana dili İngilizce olmayan öğrencilerin dil yeterliliklerini otomatik olarak analiz eden ve altı farklı kriterde puanlayan bir Doğal Dil İşleme (NLP) modeli başarıyla geliştirilmiştir.

* **Teknik Başarı:** `TfidfVectorizer` ve `MultiOutputRegressor` çatısı altında yapılandırılan `Ridge Regression` modeli, karmaşık metin yapıları üzerinde tutarlı tahminler üreterek **0.5618 MCRMSE** hata skoruna ulaşmıştır.
* **İş Akışı Otomasyonu:** Scikit-learn `Pipeline` yapısı kullanılarak, metin ön işlemeden tahmine kadar olan tüm süreç uçtan uca otomatize edilmiş ve veri sızıntısı önlenerek modelin genelleme kabiliyeti artırılmıştır.
* **Kullanıcı Deneyimi:** Sadece sayısal puanlama ile sınırlı kalınmamış; `pyspellchecker` entegrasyonu ve **Streamlit** arayüzü sayesinde öğrencilere yazım hatalarını gösteren etkileşimli ve eğitici bir geri bildirim mekanizması sunulmuştur.